# Introduction

When working with Large Language Models (LLMs) we often tend to use datasets composed by natural language text written with the latin alphabet. This has proven effective but there are some subtelties hidden in such a choice. The english language has many heteronyms, as in words that are spelled the same but have different meaning and pronounciation. Also the phonetics of a word can help relate it to other words or concepts even if their spelling could be not much alike.

The International Phonetics Alphabet (IPA) provides an extended alphabet that precisely describes how words are pronounced. The usage of such an extended alphabet, instead of the latin alphabet, could help LLMs to improve there understanding of the dataset and consequently their accuracy in inference.

The project takes a well known dataset, the [CHILDES](https://en.wikipedia.org/wiki/CHILDES) dataset, and it's IPA equivalent, [IPA-CHILDES](https://huggingface.co/datasets/phonemetransformers/IPA-CHILDES), and uses them to fine-tune two identical bert-like models to compare the results.

# Setup

In [1]:
!pip install transformers[torch]
!pip install accelerate -U

# Imports

To easily use the dataset and tokenizer from hugging face the dataset and transformers libraries are used

In [2]:
import math

import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments


# Load Dataset

In [3]:
dataset = load_dataset("phonemetransformers/IPA-CHILDES", "EnglishUK")
dataset = dataset["train"].train_test_split(test_size=0.01, seed=42)
train_data, eval_data = dataset["train"], dataset["test"]
train_data[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Eng-UK/processed.csv:   0%|          | 0.00/649M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2043115 [00:00<?, ? examples/s]

{'processed_gloss': 'mummy?',
 'ipa_transcription': 'm ʌ m i WORD_BOUNDARY',
 'character_split_utterance': 'm u m m y ? WORD_BOUNDARY',
 'is_child': True,
 'id': 9377738,
 'gloss': 'Mummy',
 'stem': 'Mummy',
 'type': 'question',
 'language': 'eng',
 'num_morphemes': 1,
 'num_tokens': 1,
 'part_of_speech': 'n:prop',
 'speaker_role': 'Target_Child',
 'target_child_age': 45.03377892769872,
 'target_child_sex': 'male',
 'collection_id': 12,
 'corpus_id': 223,
 'speaker_id': 15481,
 'target_child_id': 15481,
 'transcript_id': 24932}

From the datset only the latin alphabet representation and the IPA representations will be used

In [4]:
latin_key = "processed_gloss"
ipa_key = "ipa_transcription"

latin_train_data = train_data.select_columns([latin_key])
latin_eval_data  = eval_data.select_columns([latin_key])

ipa_train_data = train_data.select_columns([ipa_key])
ipa_eval_data  = eval_data.select_columns([ipa_key])

print(latin_train_data[:5])
print(ipa_train_data[:5])

# To decrease execution time, subsets are selected

train_range = range(20000)
eval_range = range(2000)
test_range = range(3000, 6000)

latin_train_subset = latin_train_data.select(train_range)
latin_eval_subset  = latin_eval_data.select(eval_range)
latin_test_subset = latin_eval_data.select(test_range)

ipa_train_subset = ipa_train_data.select(train_range)
ipa_eval_subset  = ipa_eval_data.select(eval_range)
ipa_test_subset = ipa_eval_data.select(test_range)



{'processed_gloss': ['mummy?', 'so are you the doctor then today?', 'tapping.', 'more bicbics.', 'all like this.']}
{'ipa_transcription': ['m ʌ m i WORD_BOUNDARY', 's əʊ WORD_BOUNDARY ɑː WORD_BOUNDARY j uː WORD_BOUNDARY ð ə WORD_BOUNDARY d ɒ kʰ tʰ ə WORD_BOUNDARY ð e n WORD_BOUNDARY tʰ ə d eɪ WORD_BOUNDARY', 'tʰ æ pʰ ɪ ŋ WORD_BOUNDARY', 'm ɔː WORD_BOUNDARY b ɪ kʰ b ɪ kʰ s WORD_BOUNDARY', 'ɔː l WORD_BOUNDARY l aɪ kʰ WORD_BOUNDARY ð ɪ s WORD_BOUNDARY']}


# Tokenizer function

Since two types of datasets are used, latin alphabet and IPA, two different tokenizers must be used

In [5]:
latin_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
ipa_tokenizer = AutoTokenizer.from_pretrained("phonemetransformers/ipa-childes-tokenizers", subfolder="EnglishUK")

def tokenize(batch, key, tokenizer):
    return tokenizer(batch[key], truncation=True, max_length=128)

def latin_tokenize(batch):
    return tokenize(batch, key=latin_key, tokenizer=latin_tokenizer)

def ipa_tokenize(batch):
    return tokenize(batch, key=ipa_key, tokenizer=ipa_tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/109 [00:00<?, ?B/s]

# Bert - IPA-Model Comparison

The dataset is featured in a [paper](https://arxiv.org/abs/2504.03036) that used it to train a GPT2 model.

A good first experiment could be to compare a BERT-like model to the paper's model

In [6]:
# Latin model
latin_model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")
collator = DataCollatorForLanguageModeling(tokenizer=latin_tokenizer, mlm_probability=0.15)

latin_eval_tok = latin_eval_subset.map(latin_tokenize, batched=True, remove_columns=latin_eval_subset.column_names)
latin_loader = DataLoader(
    latin_eval_tok, batch_size=16, shuffle=False, collate_fn=collator
)

latin_model.eval()
latin_losses = []

with torch.no_grad():
    for batch in latin_loader:
        outputs = latin_model(**batch)
        loss = outputs.loss
        if not torch.isnan(loss):
            latin_losses.append(loss.item())

latin_loss = sum(latin_losses) / len(latin_losses)
latin_ppl = math.exp(latin_loss)
latin_bpc = latin_loss / math.log(2)

print(f"DistilBERT (Latin):")
print(f"  loss = {latin_loss:.4f}")
print(f"  perplexity = {latin_ppl:.2f}")
print(f"  bits-per-character = {latin_bpc:.4f}")

# IPA model
ipa_model = AutoModelForCausalLM.from_pretrained("phonemetransformers/ipa-childes-models-medium", subfolder="EnglishUK")
ipa_collator = DataCollatorForLanguageModeling(tokenizer=ipa_tokenizer, mlm=False)

ipa_eval_tok = ipa_eval_subset.map(ipa_tokenize, batched=True, remove_columns=ipa_eval_subset.column_names)

ipa_loader = DataLoader(
    ipa_eval_tok,
    batch_size=16,
    shuffle=False,
    collate_fn=ipa_collator,
)

ipa_model.eval()
ipa_losses = []

with torch.no_grad():
    for batch in ipa_loader:
        outputs = ipa_model(**batch)
        loss = outputs.loss
        if not torch.isnan(loss):
            ipa_losses.append(loss.item())

ipa_loss = sum(ipa_losses) / len(ipa_losses)
ipa_ppl = math.exp(ipa_loss)
ipa_bpc = ipa_loss / math.log(2)

print(f"IPA GPT-2 (Phonemic):")
print(f"  loss = {ipa_loss:.4f}")
print(f"  perplexity = {ipa_ppl:.2f}")
print(f"  bits-per-character = {ipa_bpc:.4f}")

# summary
print("Summary:")
print(f"{'Model':<25} {'Loss':>10} {'Perplexity':>15} {'Bits/Char':>15}")
print("-" * 65)
print(f"{'DistilBERT (Latin)':<25} {latin_loss:>10.4f} {latin_ppl:>15.2f} {latin_bpc:>15.4f}")
print(f"{'IPA GPT-2 (Phonemic)':<25} {ipa_loss:>10.4f} {ipa_ppl:>15.2f} {ipa_bpc:>15.4f}")


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DistilBERT (Latin):
  loss = 3.6552
  perplexity = 38.68
  bits-per-character = 5.2734


config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

EnglishUK/model.safetensors:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


IPA GPT-2 (Phonemic):
  loss = 7.9249
  perplexity = 2765.21
  bits-per-character = 11.4332
Summary:
Model                           Loss      Perplexity       Bits/Char
-----------------------------------------------------------------
DistilBERT (Latin)            3.6552           38.68          5.2734
IPA GPT-2 (Phonemic)          7.9249         2765.21         11.4332


This comparison though is not fair, because of the usage of different models with different architecture and goals. Also BERT has not been fine-tuned to the IPA dataset.

To execute a valid comparison the same identical model should be used, fine-tuned on the childes/ipa-childes dataset in advance.

# BERT - IPA-BERT Comparison

To ensure the comparison is fair, a training function is defined and used for both latin and IPA

In [9]:

def train_bert_like(tokenizer, key, train, eval):
    if tokenizer.mask_token is None:
        tokenizer.add_special_tokens({"mask_token": "[MASK]"})

    def tokenize(batch):
        return tokenizer(batch[key], truncation=True, max_length=128)

    train_tok = train.map(tokenize, batched=True, remove_columns=train.column_names)
    eval_tok = eval.map(tokenize, batched=True, remove_columns=eval.column_names)

    model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")
    model.resize_token_embeddings(len(tokenizer))

    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

    training_args = TrainingArguments(
        output_dir="./ipa-bert-like",
        save_strategy="epoch",
        logging_steps=100,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=2,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=200,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        tokenizer=tokenizer,
        data_collator=collator,
    )

    trainer.train()
    return trainer



In [10]:
latin_trainer = train_bert_like(latin_tokenizer, latin_key, latin_train_subset, latin_eval_subset)
ipa_trainer = train_bert_like(ipa_tokenizer, ipa_key, ipa_train_subset, ipa_eval_subset)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

/tmp/ipython-input-2058401324.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
100,3.192300
200,2.776800
300,2.713300
400,2.631400
500,2.633100
600,2.704800
700,2.499100
800,2.629200
900,2.800900
1000,2.657100


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

/tmp/ipython-input-2058401324.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 3, 'pad_token_id': 1}.


Step,Training Loss
100,3.458400
200,3.280000
300,3.240900
400,3.235200
500,3.175800
600,3.202400
700,3.219500
800,3.196200
900,3.154900
1000,3.219300


In [11]:


latin_test_tok = latin_eval_data.map(latin_tokenize, batched=True, remove_columns=latin_eval_data.column_names)

test_range = None
max_range = 10000
range_size = 2000
step = int(range_size/2)
for start in range(0, max_range, step):
    if test_range is None:
        end = start + range_size
        latin_test_tok_subset = latin_test_tok.select(range(start, end))
        metrics = latin_trainer.evaluate(eval_dataset=latin_test_tok_subset)
        loss = metrics["eval_loss"]
        if not math.isnan(loss):
            test_range = range(start, end)
    else:
        break

loss = metrics["eval_loss"]
ppl = math.exp(loss)
bpc = loss / math.log(2)

print("\n✅ Latin DistilBERT-like model trained")
print(f"  loss = {loss:.4f}")
print(f"  perplexity = {ppl:.2f}")
print(f"  bits-per-character = {bpc:.4f}")


ipa_test_tok = ipa_eval_data.map(ipa_tokenize, batched=True, remove_columns=ipa_eval_data.column_names)

metrics = ipa_trainer.evaluate(eval_dataset=ipa_test_tok.select(test_range))
loss = metrics["eval_loss"]
ppl = math.exp(loss)
bpc = loss / math.log(2)

print("\n✅ IPA DistilBERT-like model trained")
print(f"  loss = {loss:.4f}")
print(f"  perplexity = {ppl:.2f}")
print(f"  bits-per-character = {bpc:.4f}")


Map:   0%|          | 0/20432 [00:00<?, ? examples/s]


✅ Latin DistilBERT-like model trained
  loss = 2.1424
  perplexity = 8.52
  bits-per-character = 3.0908


Map:   0%|          | 0/20432 [00:00<?, ? examples/s]


✅ IPA DistilBERT-like model trained
  loss = 2.3099
  perplexity = 10.07
  bits-per-character = 3.3324


# Conclusion

Although the IPA dataset should have provided improved context, the results are not improved, as the IPA BERT model performs slightly worse in all metrics.

Since the CHILDES dataset contains mainly short utterances the model might not have benefitted from the IPA representation as much as if the datasets phrases where larger.

Still when transcribing recorded dialogue it might be easiear to use the IPA representation. Also IPA allows models to be trained with multiple languages using the same alphabet, and consequently the same tokenizer.